In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Dropout, Input, Flatten, LSTM, GRU, Conv1D, MaxPooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, f1_score, accuracy_score,
                             matthews_corrcoef, confusion_matrix, roc_auc_score, average_precision_score)
from imblearn.over_sampling import RandomOverSampler
import time, warnings, os, random
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_seed(42)
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [2]:

df = pd.read_csv(r"C:\Users\nuzha\IDD_project\IDD_DS\nids data-20260820T135751Z-1-001\nids data\NF-CSE-CIC-IDS2018\88a47ba2ab64258e_MOHANAD_A4706\data\NF-CSE-CIC-IDS2018.csv")   # or NF-UNSW-NB15-v3.csv

df = df.fillna(0)
df.replace([np.inf, -np.inf], 0, inplace=True)

# ---------- Basic ratios (same as before) ----------
df['IN_BYTE_PER_PKT']    = df['IN_BYTES']  / (df['IN_PKTS']  + 1e-6)
df['OUT_BYTE_PER_PKT']   = df['OUT_BYTES'] / (df['OUT_PKTS'] + 1e-6)
df['TOTAL_BYTES']        = df['IN_BYTES']  + df['OUT_BYTES']
df['TOTAL_PKTS']         = df['IN_PKTS']   + df['OUT_PKTS']
df['FLOW_DUR_PER_PKT']   = df['FLOW_DURATION_MILLISECONDS'] / (df['TOTAL_PKTS'] + 1e-6)
df['IN_OUT_BYTES_RATIO'] = df['IN_BYTES'] / (df['OUT_BYTES'] + 1e-6)
df['IN_OUT_PKTS_RATIO']  = df['IN_PKTS']  / (df['OUT_PKTS']  + 1e-6)

# ---------- Rate features (important addition) ----------
df['IN_PKT_RATE']  = df['IN_PKTS']  / (df['FLOW_DURATION_MILLISECONDS'] / 1000 + 1e-6)
df['OUT_PKT_RATE'] = df['OUT_PKTS'] / (df['FLOW_DURATION_MILLISECONDS'] / 1000 + 1e-6)
df['IN_BYTE_RATE'] = df['IN_BYTES'] / (df['FLOW_DURATION_MILLISECONDS'] / 1000 + 1e-6)
df['OUT_BYTE_RATE']= df['OUT_BYTES']/ (df['FLOW_DURATION_MILLISECONDS'] / 1000 + 1e-6)

# ---------- TCP flags ----------
df['TCP_SYN'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x02 else 0)
df['TCP_ACK'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x10 else 0)
df['TCP_FIN'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x01 else 0)
df['TCP_RST'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x04 else 0)
df['TCP_PSH'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x08 else 0)

# ---------- Protocol / Application indicators ----------
# These columns usually exist in NF-CSE-CIC-IDS2018
if 'PROTOCOL' in df.columns:
    df['PROTOCOL'] = df['PROTOCOL'].astype(float)

if 'L7_PROTO' in df.columns:
    df['L7_PROTO'] = df['L7_PROTO'].astype(float)

# Simple binary indicators (common in the PLOS notebook)
df['IS_HTTP']  = ((df.get('L7_PROTO', 0) == 7) | (df.get('L4_DST_PORT', 0) == 80)).astype(int)
df['IS_HTTPS'] = ((df.get('L7_PROTO', 0) == 91) | (df.get('L4_DST_PORT', 0) == 443)).astype(int)
df['IS_DNS']   = ((df.get('L7_PROTO', 0) == 5) | (df.get('L4_DST_PORT', 0) == 53)).astype(int)

# ---------- Label encoding ----------
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])

# ---------- Final feature list ----------
exclude = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Label', 'Attack']
feature_cols = [c for c in df.columns if c not in exclude]

print("Number of features:", len(feature_cols))
print(feature_cols)

Number of features: 29
['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS', 'TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'IN_BYTE_PER_PKT', 'OUT_BYTE_PER_PKT', 'TOTAL_BYTES', 'TOTAL_PKTS', 'FLOW_DUR_PER_PKT', 'IN_OUT_BYTES_RATIO', 'IN_OUT_PKTS_RATIO', 'IN_PKT_RATE', 'OUT_PKT_RATE', 'IN_BYTE_RATE', 'OUT_BYTE_RATE', 'TCP_SYN', 'TCP_ACK', 'TCP_FIN', 'TCP_RST', 'TCP_PSH', 'IS_HTTP', 'IS_HTTPS', 'IS_DNS']


In [3]:
zero_day_attacks = [
    'DoS attacks-GoldenEye',
    'DoS attacks-Hulk',
    'DoS attacks-SlowHTTPTest',
    'SSH-Bruteforce'
]

df = df.reset_index(drop=True)

zero_day_df = df[df['Attack'].isin(zero_day_attacks)].copy()
normal_df   = df[~df['Attack'].isin(zero_day_attacks)].copy()

cutoff1 = int(0.80 * len(normal_df))
dev_df      = normal_df.iloc[:cutoff1].copy()
test_normal = normal_df.iloc[cutoff1:].copy()

cutoff2 = int(0.90 * len(dev_df))
train_df = dev_df.iloc[:cutoff2].copy()
val_df   = dev_df.iloc[cutoff2:].copy()

print("Train     :", train_df.shape)
print("Validation:", val_df.shape)
print("Known Test:", test_normal.shape)
print("Zero-day  :", zero_day_df.shape)
print("Zero-day leakage (must be 0):",
      train_df[train_df['Attack'].isin(zero_day_attacks)].shape[0],
      val_df[val_df['Attack'].isin(zero_day_attacks)].shape[0])

Train     : (5797171, 33)
Validation: (644131, 33)
Known Test: (1610326, 33)
Zero-day  : (340773, 33)
Zero-day leakage (must be 0): 0 0


In [4]:
# Exact names from your value_counts
golden_df = zero_day_df[zero_day_df['Attack'] == 'DoS attacks-GoldenEye'].copy()
hulk_df   = zero_day_df[zero_day_df['Attack'] == 'DoS attacks-Hulk'].copy()
slow_df   = zero_day_df[zero_day_df['Attack'] == 'DoS attacks-SlowHTTPTest'].copy()
ssh_df    = zero_day_df[zero_day_df['Attack'] == 'SSH-Bruteforce'].copy()

print("Golden :", len(golden_df))
print("Hulk   :", len(hulk_df))
print("Slow   :", len(slow_df))
print("SSH    :", len(ssh_df))

Golden : 32850
Hulk   : 108136
Slow   : 105550
SSH    : 94237


In [5]:
def create_sequences(data, feature_cols, window_size=10, stride=5):
    sequences, labels, attacks = [], [], []
    feats = data[feature_cols].values.astype(np.float32)
    labs  = data['Label'].values
    atks  = data['Attack'].values
    for i in range(0, len(feats) - window_size + 1, stride):
        sequences.append(feats[i:i+window_size])
        labels.append(labs[i + window_size - 1])
        attacks.append(atks[i + window_size - 1])
    return np.array(sequences, dtype=np.float32), np.array(labels), np.array(attacks)

def safe_create_sequences(data, feature_cols, window_size=10, stride=5):
    if len(data) < window_size:
        return (np.empty((0, window_size, len(feature_cols)), dtype=np.float32),
                np.empty((0,), dtype=np.int64),
                np.empty((0,), dtype=object))
    return create_sequences(data, feature_cols, window_size, stride)

WINDOW, STRIDE = 10, 5

X_train, y_train, _ = create_sequences(train_df, feature_cols, WINDOW, STRIDE)
X_val,   y_val,   _ = create_sequences(val_df,   feature_cols, WINDOW, STRIDE)
X_test_known, y_test_known, atk_known = create_sequences(test_normal, feature_cols, WINDOW, STRIDE)

X_golden, y_golden, atk_golden = safe_create_sequences(golden_df, feature_cols, WINDOW, STRIDE)
X_hulk,   y_hulk,   atk_hulk   = safe_create_sequences(hulk_df,   feature_cols, WINDOW, STRIDE)
X_slow,   y_slow,   atk_slow   = safe_create_sequences(slow_df,   feature_cols, WINDOW, STRIDE)
X_ssh,    y_ssh,    atk_ssh    = safe_create_sequences(ssh_df,    feature_cols, WINDOW, STRIDE)

print("Golden sequences:", X_golden.shape)
print("Hulk sequences  :", X_hulk.shape)
print("Slow sequences  :", X_slow.shape)
print("SSH sequences   :", X_ssh.shape)

X_test = np.concatenate([X_test_known, X_golden, X_hulk, X_slow, X_ssh], axis=0)
y_test = np.concatenate([y_test_known, y_golden, y_hulk, y_slow, y_ssh], axis=0)
attack_test = np.concatenate([atk_known, atk_golden, atk_hulk, atk_slow, atk_ssh], axis=0)

print("Final Test shape:", X_test.shape)

Golden sequences: (6569, 10, 29)
Hulk sequences  : (21626, 10, 29)
Slow sequences  : (21109, 10, 29)
SSH sequences   : (18846, 10, 29)
Final Test shape: (390214, 10, 29)


In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_val   = scaler.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape)
X_test  = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

n_s, t, f = X_train.shape
X_train_2d, y_train = RandomOverSampler(random_state=42).fit_resample(X_train.reshape(n_s, -1), y_train)
X_train = X_train_2d.reshape(-1, t, f)

n_features = X_train.shape[-1]
print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
print("n_features =", n_features)

Train: (2288644, 10, 29) | Val: (128825, 10, 29) | Test: (390214, 10, 29)
n_features = 29


In [7]:
class StandardESN(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1,1,(self.units,self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units,self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        super().build(input_shape)
    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))
        def body(t, state):
            pre = tf.matmul(inputs[:,t,:], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t+1, (1-self.leak_rate)*state + self.leak_rate*tf.tanh(pre)
        _, final = tf.while_loop(lambda t,s: t < tf.shape(inputs)[1], body, (0, state))
        return final

class HarmonicESN(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1,1,(self.units,self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units,self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        super().build(input_shape)
    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))
        def body(t, state):
            pre = tf.matmul(inputs[:,t,:], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t+1, (1-self.leak_rate)*state + self.leak_rate*tf.sin(pre)
        _, final = tf.while_loop(lambda t,s: t < tf.shape(inputs)[1], body, (0, state))
        return final

class FB_HESN_Layer(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, omega_min=0.3, omega_max=3.5, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
        self.omega_min = omega_min
        self.omega_max = omega_max
    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1,1,(self.units,self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units,self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        omega = np.random.uniform(self.omega_min, self.omega_max, self.units).astype(np.float32)
        self.omega = self.add_weight(shape=(self.units,), initializer=tf.constant_initializer(omega), trainable=False)
        super().build(input_shape)
    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))
        def body(t, state):
            pre = tf.matmul(inputs[:,t,:], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t+1, (1-self.leak_rate)*state + self.leak_rate*tf.sin(self.omega * pre)
        _, final = tf.while_loop(lambda t,s: t < tf.shape(inputs)[1], body, (0, state))
        return final

def evaluate_and_store(y_true, y_pred, y_prob, attack_test, model, model_name, results_list):
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average='macro')
    mcc  = matthews_corrcoef(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    roc = roc_auc_score(y_true, y_prob)
    pr  = average_precision_score(y_true, y_prob)

    # CSE zero-day names
    zero_days = ['DoS attacks-GoldenEye', 'DoS attacks-Hulk', 
                 'DoS attacks-SlowHTTPTest', 'SSH-Bruteforce']
    
    zd = {}
    for atk in zero_days:
        mask = np.array([str(a) == atk for a in attack_test])
        total = mask.sum()
        detected = (y_pred[mask] == 1).sum() if total > 0 else 0
        zd[atk] = (detected, total, detected/total if total > 0 else np.nan)

    valid = [zd[a][2] for a in zd if not np.isnan(zd[a][2])]
    zd_macro = np.mean(valid) if valid else np.nan
    zd_min   = np.min(valid) if valid else np.nan

    total_params = model.count_params()
    trainable = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))

    _ = model.predict(X_test[:128], verbose=0)
    t0 = time.time()
    _ = model.predict(X_test, batch_size=256, verbose=0)
    infer_time = time.time() - t0

    results_list.append({
        'Model': model_name,
        'Accuracy': round(acc,4),
        'Macro-F1': round(f1,4),
        'MCC': round(mcc,4),
        'FPR': round(fpr,4),
        'FNR': round(fnr,4),
        'ROC-AUC': round(roc,4),
        'PR-AUC': round(pr,4),
        'GoldenEye': f"{zd['DoS attacks-GoldenEye'][0]}/{zd['DoS attacks-GoldenEye'][1]} ({zd['DoS attacks-GoldenEye'][2]:.4f})",
        'Hulk': f"{zd['DoS attacks-Hulk'][0]}/{zd['DoS attacks-Hulk'][1]} ({zd['DoS attacks-Hulk'][2]:.4f})",
        'SlowHTTPTest': f"{zd['DoS attacks-SlowHTTPTest'][0]}/{zd['DoS attacks-SlowHTTPTest'][1]} ({zd['DoS attacks-SlowHTTPTest'][2]:.4f})",
        'SSH-Bruteforce': f"{zd['SSH-Bruteforce'][0]}/{zd['SSH-Bruteforce'][1]} ({zd['SSH-Bruteforce'][2]:.4f})",
        'ZD-MacroRecall': round(zd_macro,4),
        'ZD-MinRecall': round(zd_min,4),
        'Total Params': total_params,
        'Trainable Params': trainable,
        'Inference Time (s)': round(infer_time,2)
    })
    print(f"{model_name} done | Macro-F1={f1:.4f} | ZD-Macro={zd_macro:.4f}")

In [8]:
results = []

def train_neural(build_fn, name):
    tf.keras.backend.clear_session()
    set_seed(42)
    model = build_fn()
    model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=40, batch_size=256,
              callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)],
              verbose=1)
    y_prob = model.predict(X_test, batch_size=256, verbose=0)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    evaluate_and_store(y_test, y_pred, y_prob, attack_test, model, name, results)

def build_standard_esn():
    inputs = Input(shape=(10, n_features))
    x = StandardESN(units=64)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_harmonic_esn():
    inputs = Input(shape=(10, n_features))
    x = HarmonicESN(units=64)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_fb_hesn():
    inputs = Input(shape=(10, n_features))
    x = FB_HESN_Layer(units=64, spectral_radius=0.9, leak_rate=0.3,
                      omega_min=0.3, omega_max=1.0)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_gru():
    inputs = Input(shape=(10, n_features))
    x = GRU(64)(inputs)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

# Run
train_neural(build_standard_esn, "Standard ESN-64")
train_neural(build_harmonic_esn, "Harmonic ESN-64")
train_neural(build_fb_hesn, "FB-HESN-64")
train_neural(build_gru, "GRU-64")

df_master = pd.DataFrame(results)
print(df_master.to_string(index=False))
df_master.to_csv("CSE_master_64.csv", index=False)


Epoch 1/40
8941/8941 ━━━━━━━━━━━━━━━━━━━━ 63s 7ms/step - accuracy: 0.7463 - loss: 0.4498 - val_accuracy: 0.6639 - val_loss: 0.4381
Epoch 2/40
8941/8941 ━━━━━━━━━━━━━━━━━━━━ 78s 6ms/step - accuracy: 0.7601 - loss: 0.4257 - val_accuracy: 0.6699 - val_loss: 0.4239
Epoch 3/40
8941/8941 ━━━━━━━━━━━━━━━━━━━━ 50s 6ms/step - accuracy: 0.7630 - loss: 0.4197 - val_accuracy: 0.6613 - val_loss: 0.4279
Epoch 4/40
8941/8941 ━━━━━━━━━━━━━━━━━━━━ 48s 5ms/step - accuracy: 0.7649 - loss: 0.4161 - val_accuracy: 0.6618 - val_loss: 0.4284
Epoch 5/40
8941/8941 ━━━━━━━━━━━━━━━━━━━━ 36s 4ms/step - accuracy: 0.7659 - loss: 0.4136 - val_accuracy: 0.6568 - val_loss: 0.4247
Epoch 6/40
8941/8941 ━━━━━━━━━━━━━━━━━━━━ 40s 4ms/step - accuracy: 0.7673 - loss: 0.4113 - val_accuracy: 0.6502 - val_loss: 0.4321
Epoch 7/40
8941/8941 ━━━━━━━━━━━━━━━━━━━━ 37s 4ms/step - accuracy: 0.7674 - loss: 0.4102 - val_accuracy: 0.6547 - val_loss: 0.4260
Standard ESN-64 done | Macro-F1=0.3148 | ZD-Macro=0.0073
Epoch 1/40
8941/8941 ━━━━